# 🔧 ***Fine-Tuning Qwen2.5-Omni with LLaMA-Factory***

This notebook demonstrates how to fine-tune the **Qwen2.5-Omni** model for **Alzheimer’s Disease (AD) speech classification** using the [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory) framework. Unlike the PyTorch-based finetune script, here the entire training is configured via a **YAML file** and launched with the `llamafactory-cli`.

## 📂 Expected Setup
- **Training and Evaluation CSV**: must contain `uid`, `transcription`, `label`.  
- **Audio Directories**: should contain `.wav`/`.flac` files named with the corresponding `uid`.  
- **Model**: [`Qwen2.5-Omni`](https://huggingface.co/Qwen/Qwen2.5-Omni-7B).  
- **Config file (`config_train.yaml`)**: defines model, dataset paths, training args, evaluation, and logging.  


## ***Installing LLaMA-Factory***

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e ".[torch,metrics]" --no-build-isolation

## ***Converting Datasets to Standard JSON Format***

In [ ]:
import os
import json
import pandas as pd


def create_dataset_json(train_csv_path, train_audio_path):
    # Read the CSV file
    df = pd.read_csv(train_csv_path)
    
    # Create the dataset list
    dataset = []
    
    # Define the question based on the provided template
    question = (
        "Transcription: \"{transcription}\"\n\n"
        "Based on the speech audio and its transcription, "
        "classify the speaker's cognitive status with a single word "
        "from the following options: 'dementia' (Alzheimer's Disease and Related Dementia), "
        "or 'control' (Cognitively Normal)."
    )
    
    for _, row in df.iterrows():
        uid = row['uid']
        transcription = row['transcription']
        
        # Determine the answer based on diagnosis columns
        answer = row['label']
        
        # Format the question with the transcription
        formatted_question = question.format(transcription=transcription)
        
        # Create the entry following the mllm_audio_demo.json format
        entry = {
            "messages": [
                {
                    "content": f"<audio>{formatted_question}",
                    "role": "user"
                },
                {
                    "content": answer,
                    "role": "assistant"
                }
            ],
            "audios": [
                os.path.join(train_audio_path, f"{uid}.wav")
            ]
        }
        
        dataset.append(entry)
    
    # Write to JSON file
    with open('LLaMA-Factory/data/ad_detection_dataset.json', 'w', encoding='utf-8') as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)
    
    print(f"Created dataset with {len(dataset)} entries")
    print(f"Saved to: LLaMA-Factory/data/ad_detection_dataset.json")
    
    # Print some statistics
    answers = [entry['messages'][1]['content'] for entry in dataset]
    print(f"Total entries: {len(answers)}")

create_dataset_json() 

## ***Adding Created Datasets to LLaMA-Factory's Framework***

Add this snippet to the end of `LLaMA-Factory/data/dataset_info.json` in order for the framework to recognize the dataset.

```json
"ad_detection_dataset": {
    "file_name": "ad_detection_dataset.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "messages",
      "audios": "audios"
    },
    "tags": {
      "role_tag": "role",
      "content_tag": "content",
      "user_tag": "user",
      "assistant_tag": "assistant"
    }
  },
```

## ***Fine-tuning the Model***

You may change training configuration according to the unique aspects of the dataset and task via the <code>config_train.yaml</code> file. 

In [ ]:
!llamafactory-cli train config_train.yaml

# 🧪 ***Testing Fine-Tuned Qwen2.5-Omni***

This section of the notebook provides a **testing pipeline** for evaluating a fine-tuned **Qwen2.5-Omni** model on a **2-label cognitive status classification task**:  
- `dementia` → speakers with Alzheimer's Disease or related dementia  
- `control` → cognitively normal speakers  

The script loads the fine-tuned model, processes the **test CSV + audio files**, generates predictions, and computes detailed evaluation metrics.  

---

## 🔑 Key Features
- **Custom Tester Class**: `AudioClassificationTester2Label`  
  - Loads a fine-tuned Qwen2.5-Omni model + processor.  
  - Reads a CSV with `uid`, `transcription`, `label`.  
  - Constructs conversation-style inputs (audio + text prompt).  
  - Runs classification (`dementia` vs `control`) using greedy decoding.  
- **Prediction Handling**:  
  - Extracts cognitive label from model response.  
  - Maps unknown or malformed outputs to `"unknown"`.  
- **Evaluation Metrics**:  
  - Accuracy  
  - Weighted Precision, Recall, and F1  
  - Macro Precision, Recall, and F1  
  - Per-class Precision, Recall, F1  
  - Confusion Matrix  
- **Result Saving**:  
  - `detailed_results.csv` (per-sample predictions)  
  - `test_results_2label.xlsx` with multiple sheets:  
    - Detailed results  
    - Valid predictions only  
    - Summary metrics  
    - Confusion matrix  
    - Per-class breakdown  

---

## 📂 Expected Inputs
- **Fine-tuned Model Path** (`--model_path`) → directory containing the trained Qwen2.5-Omni weights.  
- **Test CSV** (`--test_csv`) → contains columns:  
  - `uid` → sample ID (used to find corresponding audio file)  
  - `transcription` → text transcript of the audio  
  - `label` → ground truth label (0 = control, 1 = dementia)  
- **Audio Files Directory** (`--audio_path`) → contains `.wav` files named after the `uid`.  

---

## ⚙️ Main Arguments
- `--model_path` → path to the fine-tuned Qwen2.5-Omni model.  
- `--test_csv` → path to test CSV file.  
- `--audio_path` → directory containing test audio files.  
- `--num_samples` → (optional) limit number of test samples.  
- `--start_idx` → (optional) start index for evaluation subset.  
- `--output_path` → directory to save results (CSV + Excel).  

---

## 🚀 Workflow
1. **Initialize tester** → loads model, processor, and test dataset.  
2. **Predict labels** → runs `tester.test_subset()` on test samples.  
3. **Evaluate results** → calculates metrics and confusion matrix.  
4. **Save outputs** → results saved in CSV + Excel for inspection.  

## ***Installing Requirements***

In [ ]:
!pip install -r test_requirements.txt
!pip install git+https://github.com/Kuangdd01/transformers.git@qwen25omni

## ***Model Inference***

In [ ]:
!python test_finetuned_model.py \
    --model_path "path/to/your/fine-tuned/model" \
    --test_csv "path/to/your/test.csv" \
    --audio_path "path/to/your/audio_files" \
    --output_path "path/to/save/results"